# DeepIRI ZepGPU - GPU Task Demo

This notebook demonstrates how to submit and monitor GPU tasks using the DeepIRI ZepGPU framework.

In [ ]:
import numpy as np
import asyncio
from deepiri_zepgpu import submit_task, TaskSubmitter

## Simple GPU Task

In [ ]:
def matrix_multiply(A, B):
    """Matrix multiplication on GPU."""
    import numpy as np
    return np.matmul(A, B)

# Create sample matrices
A = np.random.randn(1000, 500).astype(np.float32)
B = np.random.randn(500, 1000).astype(np.float32)

# Submit task
task_id = submit_task(matrix_multiply, A, B, gpu_memory_mb=1024)
print(f"Task submitted: {task_id}")

## Wait for Result

In [ ]:
# Submit and wait for result
result = submit_task(matrix_multiply, A, B, wait=True, timeout_seconds=60)
print(f"Result shape: {result.shape}")

## Using TaskSubmitter

In [ ]:
async def main():
    async with TaskSubmitter() as submitter:
        # Submit multiple tasks
        task_ids = []
        for i in range(3):
            task_id = await submitter.submit(
                matrix_multiply, A, B,
                priority=TaskPriority.HIGH if i == 0 else TaskPriority.NORMAL
            )
            task_ids.append(task_id)
            print(f"Submitted: {task_id}")
        
        # Monitor tasks
        for task_id in task_ids:
            task = submitter.get_task(task_id)
            print(f"Task {task_id[:8]}: {task.status}")

asyncio.run(main())

## Monte Carlo Simulation

In [ ]:
def estimate_pi(num_samples):
    """Estimate Pi using Monte Carlo."""
    x = np.random.uniform(0, 1, num_samples)
    y = np.random.uniform(0, 1, num_samples)
    inside = np.sum(x**2 + y**2 <= 1)
    return 4 * inside / num_samples

# Submit Monte Carlo task
task_id = submit_task(estimate_pi, 10000000, gpu_memory_mb=512)
print(f"Monte Carlo task submitted: {task_id}")